Importing packages

In [ ]:
import pandas as pd

Data load

In [ ]:
user_path = "Uresr/piotr/"

In [ ]:
snapshot_bak = pd.read_csv(f"{user_path}/UEP-MachineLearning/data_collected/markets_snapshots.bak.csv", on_bad_lines="warn")
snapshot_regular = pd.read_csv(f"{user_path}/UEP-MachineLearning/data_collected/markets_snapshots.csv", on_bad_lines="warn")
static_bak = pd.read_csv(f"{user_path}/UEP-MachineLearning/data_collected/markets_static.bak.csv", on_bad_lines="warn")
static_regular = pd.read_csv(f"{user_path}/UEP-MachineLearning/data_collected/markets_static.csv", on_bad_lines="warn")

/var/folders/dk/zjwq8trx6cbbykf77jz61hnc0000gn/T/ipykernel_63937/1093181284.py:2: DtypeWarning: Columns (0: restricted, 1: promotion_group) have mixed types. Specify dtype option on import or set low_memory=False.
  snapshot_regular = pd.read_csv("/Users/piotr/UEP-MachineLearning/data_collected/markets_snapshots.csv", on_bad_lines="warn")
/var/folders/dk/zjwq8trx6cbbykf77jz61hnc0000gn/T/ipykernel_63937/1093181284.py:3: ParserWarning: Skipping line 3423: expected 11 fields, saw 12
Skipping line 3424: expected 11 fields, saw 12
Skipping line 3425: expected 11 fields, saw 12
Skipping line 3426: expected 11 fields, saw 12
Skipping line 3427: expected 11 fields, saw 12
Skipping line 3428: expected 11 fields, saw 12
Skipping line 3429: expected 11 fields, saw 12
Skipping line 3430: expected 11 fields, saw 12
Skipping line 3431: expected 11 fields, saw 12
Skipping line 3432: expected 11 fields, saw 12
Skipping line 3433: expected 11 fields, saw 12
Skipping line 3434: expected 11 fields, saw 1

Cleaning data collected

In [19]:
snapshot_key_cols = ['snapshot_ts', 'market_id']

In [20]:
snapshot_combined = pd.concat([snapshot_bak, snapshot_regular], ignore_index=True)

In [21]:
snapshot_combined = snapshot_combined.drop_duplicates(subset=snapshot_key_cols, keep='first')
print(f"Snapshot combined: {snapshot_combined.shape[0]} rows (deduplicated)")

Snapshot combined: 652220 rows (deduplicated)


In [22]:
static_key_cols = ['market_id']

In [23]:
static_combined = pd.concat([static_bak, static_regular], ignore_index=True)

In [24]:
static_combined = static_combined.drop_duplicates(subset=static_key_cols, keep='first')
print(f"Static combined: {static_combined.shape[0]} rows (deduplicated)")

Static combined: 16660 rows (deduplicated)


In [25]:
final_data = snapshot_combined.merge(static_combined, on='market_id', how='left')

print(f"Final merged data: {final_data.shape[0]} rows, {final_data.shape[1]} columns")

Final merged data: 652220 rows, 49 columns


In [ ]:
db1_merged = final_data.copy()
db1_merged.to_csv(f"{user_path}/UEP-MachineLearning/data_model/db_collected.csv")

Cleaning data enriched

In [ ]:
p_orderFilled = pd.read_csv('{user_path}/UEP-MachineLearning/data_enriched/p_orderFilled.csv')
p_markets = pd.read_csv('{user_path}/UEP-MachineLearning/data_enriched/p_markets.csv')
p_trades = pd.read_csv('{user_path}/UEP-MachineLearning/data_enriched/p_trades.csv')

In [28]:
p_trades['timestamp'] = pd.to_datetime(p_trades['timestamp'])

In [29]:
p_snapshot_agg = p_trades.groupby('market_id').agg({
    'timestamp': 'max',
    'price': 'last',
    'usd_amount': 'sum',
    'transactionHash': 'count'
}).reset_index()

In [30]:
db2_snapshot = pd.DataFrame()
db2_snapshot['snapshot_ts'] = p_snapshot_agg['timestamp']
db2_snapshot['market_id'] = p_snapshot_agg['market_id']
db2_snapshot['active'] = True
db2_snapshot['closed'] = False
db2_snapshot['featured'] = False
db2_snapshot['new'] = False
db2_snapshot['archived'] = False
db2_snapshot['restricted'] = False
db2_snapshot['volume24hr'] = None
db2_snapshot['volume1wk'] = None
db2_snapshot['volume1mo'] = None
db2_snapshot['volume1yr'] = None
db2_snapshot['volume_total'] = p_snapshot_agg['usd_amount']
db2_snapshot['liquidity'] = None
db2_snapshot['liquidityAmm'] = None
db2_snapshot['liquidityClob'] = None
db2_snapshot['openInterest'] = None
db2_snapshot['lastTradePrice'] = p_snapshot_agg['price']
db2_snapshot['bestBid'] = None
db2_snapshot['bestAsk'] = None
db2_snapshot['spread'] = None
db2_snapshot['oneDayPriceChange'] = None
db2_snapshot['oneWeekPriceChange'] = None
db2_snapshot['commentCount'] = p_snapshot_agg['transactionHash']
db2_snapshot['competitive'] = None
db2_snapshot['score'] = None
db2_snapshot['mailchimp_tag'] = None
db2_snapshot['promotion_group'] = None

In [31]:
db2_static = pd.DataFrame()
db2_static['market_id'] = p_markets['id']
db2_static['event_id'] = None
db2_static['event_slug'] = p_markets['market_slug']
db2_static['event_title'] = p_markets['question']
db2_static['event_subtitle'] = None
db2_static['event_description'] = None
db2_static['market_slug'] = p_markets['market_slug']
db2_static['market_title'] = p_markets['question']
db2_static['market_subtitle'] = p_markets['answer1'] + ' / ' + p_markets['answer2']
db2_static['question'] = p_markets['question']
db2_static['category'] = None
db2_static['subcategory'] = None
db2_static['created_at'] = p_markets['createdAt']
db2_static['end_date'] = p_markets['closedTime']
db2_static['resolution_source'] = None
db2_static['resolved_by'] = None
db2_static['final_volume_total'] = p_markets['volume']
db2_static['final_volume24hr'] = None
db2_static['tags'] = None
db2_static['mailchimp_tag'] = None
db2_static['promotion_group'] = None
db2_static['commentCount'] = None

In [32]:
db2_merged = db2_snapshot.merge(db2_static, on='market_id', how='left')

Verify integrity

In [33]:
all_columns = list(set(db1_merged.columns) | set(db2_merged.columns))
all_columns.sort()

for col in all_columns:
    if col not in db1_merged.columns:
        db1_merged[col] = None
    if col not in db2_merged.columns:
        db2_merged[col] = None

db1_merged = db1_merged[all_columns]
db2_merged = db2_merged[all_columns]

print("\n=== SCHEMA ALIGNMENT ===")
print(f"Database 1 columns: {len(db1_merged.columns)}")
print(f"Database 2 columns: {len(db2_merged.columns)}")
print(f"Columns match: {list(db1_merged.columns) == list(db2_merged.columns)}")


=== SCHEMA ALIGNMENT ===
Database 1 columns: 49
Database 2 columns: 49
Columns match: True


In [ ]:
db2_merged.to_csv(f"{user_path}/UEP-MachineLearning/data_model/db_additional.csv")